<a href="https://colab.research.google.com/github/Pranayshukla0610/ML-projects-portfolio/blob/main/Goldman_Sachs_Financial_Intelligence_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [145]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from bokeh.io import (
    output_notebook,
    show,
    curdoc,
    output_file,
    export_png
)

from bokeh.plotting import figure
from bokeh.layouts import (
    row,
    column
)
from google.colab import files
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    Div,
    Select,
    DateRangeSlider
)

In [146]:
output_notebook()

In [147]:
BACKGROUND = '#0F172A'
CARD = '#1E293B'
TEXT = '#F8FAFC'
GRID = '#334155'
WIDTH = 650
HEIGHT = 400

In [148]:
companies = {
    "Goldman Sachs":"GS",
    "JP Morgan":"JPM",
    "Morgan Stanley":"MS",
    "Bank of America":"BAC",
    "Citigroup":"C"
}

In [149]:
ticker = companies['Goldman Sachs']

In [150]:
import yfinance as yf

data = yf.download(
    ticker,
    period='2y',
    interval = '1d'
)

/tmp/ipykernel_21826/256073594.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed


In [151]:
data.head()

Price,Close,High,Low,Open,Volume
Ticker,GS,GS,GS,GS,GS
Date,,,,,
2024-05-13,434.763489,438.252653,434.533444,437.361172,1579600
2024-05-14,439.470001,442.633230,435.194842,435.990432,2412900
2024-05-15,446.774170,446.870032,441.847198,442.460660,2217700
2024-05-16,445.269226,448.844657,443.122066,445.125448,2070300
2024-05-17,448.336670,449.237717,445.010473,447.052206,1655900


In [152]:
data.reset_index(inplace=True)

In [153]:
data.columns = data.columns.get_level_values(0)

In [154]:
data['Returns'] = (
    data['Close'].pct_change()
)

In [155]:
data['MA20'] = (
    data['Close'].rolling(20).mean()
)

data['MA50'] = (
    data['Close'].rolling(50).mean()
)

data['MA100'] = (
    data['Close'].rolling(100).mean()
)

In [156]:
data['Volatility'] = (
    data['Returns'].rolling(20).std()
)

In [157]:
#RSI

# RSI Calculation

delta = data['Close'].diff()

# Positive price movement
gain = delta.where(
    delta > 0,
    0
)

# Negative price movement
loss = -delta.where(
    delta < 0,
    0
)

# Average gain
avg_gain = gain.rolling(14).mean()

# Average loss
avg_loss = loss.rolling(14).mean()

# Relative Strength
rs = avg_gain / avg_loss

# RSI
data['RSI'] = (

    100 - (100 / (1 + rs))

)

In [158]:
investment = 10000

shares = (
    investment
    /
    data['Close'].iloc[0]
)
data['Portfolio'] = (
    shares * data['Close']
)

In [159]:
data.dropna(inplace=True)

In [160]:
source = ColumnDataSource(data)

In [161]:
latest_price = round(
    data['Close'].iloc[-1],
    2
)

daily_return = round(
    data['Returns'].iloc[-1] * 100,
    2
)

volatility = round(
    data['Volatility'].mean() * 100,
    2
)

portfolio_value = round(
    data['Portfolio'].iloc[-1],
    2
)

profit = round(
    portfolio_value - investment,
    2
)

In [162]:
risk_free_rate = 0.02

sharpe_ratio = round(

    (
        data['Returns'].mean()*252
        - risk_free_rate
    )

    /

    (
        data['Returns'].std()
        * np.sqrt(252)
    ),

    2
)

In [163]:
def create_kpi(title, value, color):
  return Div(
      text=f"""
      <div style="
      background:{color};
      padding:20px;
      border-radius:18px;
      text-align:center;
      width:220px;
      height:120px;
      box-shadow:0px 4px 15px rgba(0,0,0,0.5);
      ">

      <h2 style="
      color:white;
      ">

      {title}

      </h2>
      <h1 style="
      color:white;
      ">

      {value}

      </h1>
      </div>

 """ )

In [164]:
kpi = create_kpi(
    "Live Price",
    f"${latest_price}",
    "#2563EB"
)

kpi2 = create_kpi(
    "Daily Return",
    f"{daily_return}%",
    "#059669"
)

kpi3 = create_kpi(
    "Volatility",
    f"{volatility}%",
    '#DC2626'
)

kpi4 = create_kpi(
    'Sharpe Ratio',
    sharpe_ratio,
    "#0891b2"
)

kpi5 = create_kpi(
    'Portfolio Value',
    f"${portfolio_value}",
    "#7C3AED"
)

kpi6 = create_kpi(
    'Profit/Loss',
    f"${profit}",
    "#EA580C"
)

In [165]:
company_selector = Select(
    title = 'Select Investment Bank',
    value = 'Goldman Sachs',
    options = list(companies.keys())
)

In [166]:
date_slider = DateRangeSlider(
    title = 'Select Date Range',

    start = data['Date'].min(),

    end = data['Date'].max(),

    value = (
        data['Date'].min(),
        data['Date'].max()
    ),
    width=500
)

In [167]:
inc = data['Close'] > data['Open']

dec = data['Open'] > data['Close']

w = 12 * 60 * 60 * 1000

In [168]:
p1 = figure(
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title = 'Goldman Sachs Candlestick Analytics',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND
)

In [169]:
p1.segment(

    data['Date'],
    data['High'],

    data['Date'],
    data['Low'],

    color="white"

)

GlyphRenderer(id='p2358', ...)

In [170]:
p1.vbar(
    x = data.loc[inc,'Date'],
    width = w,
    top = data.loc[inc,'Close'],
    bottom = data.loc[inc,'Open'],
    fill_color = '#10B981',
    line_color = '#10B981'
)

GlyphRenderer(id='p2369', ...)

In [171]:
p1.vbar(
    x = data.loc[dec,'Date'],
    width = w,
    top = data.loc[dec,'Open'],
    bottom = data.loc[dec,'Close'],
    fill_color = '#EF4444',
    line_color = '#EF4444'
)

GlyphRenderer(id='p2380', ...)

In [172]:
p2 = figure (
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title = 'Moving Average Analytics',
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [173]:
p2.line(
    data['Date'],
    data['Close'],
    line_width=2,
    color='white',
    legend_label='Close Price'
)

GlyphRenderer(id='p2440', ...)

In [174]:
p2.line(
    data['Date'],
    data['MA20'],
    line_width=2,
    color='#3B82F6',
    legend_label='MA20'
)

GlyphRenderer(id='p2453', ...)

In [175]:
p2.line(
    data['Date'],
    data['MA50'],
    line_width=3,
    color='#F59E0B',
    legend_label='MA50'
)

GlyphRenderer(id='p2465', ...)

In [176]:
p2.line(
    data['Date'],
    data['MA100'],
    line_width=3,
    color='#8B5CF6',
    legend_label='MA100'
)

GlyphRenderer(id='p2477', ...)

In [177]:
p3 = figure(
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title='Trading Volume Analytics',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND
)

In [178]:
p3.vbar(
    x = data['Date'],
    top = data['Volume'],
    width=w,
    color='#06B6D4'
)

GlyphRenderer(id='p2538', ...)

In [179]:
p4 = figure(
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title='RSI Analytics',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND

)

In [180]:
p4.line(
    data['Date'],
    [70]*len(data),
    color='red',
    line_dash='dashed'
)

p4.line(
    data['Date'],
    [30]*len(data),
    color='green',
    line_dash = 'dashed'
)

GlyphRenderer(id='p2607', ...)

In [181]:
p5 = figure(
    x_axis_type='datetime',
    width = WIDTH,
    height = HEIGHT,
    title = 'Portfolio Growth Dashboard',
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [182]:
p5.line(
    data['Date'],
    data['Portfolio'],
    line_width=2,
    color='#22C55E'
)

GlyphRenderer(id='p2667', ...)

In [183]:
hist, edges = np.histogram(
    data['Returns'].dropna(),
    bins=40
)

In [184]:
p6 = figure(
    width = WIDTH,
    height = HEIGHT,
    title = "Return Distribution Analytics",
    background_fill_color = CARD,
    border_fill_color = BACKGROUND
)

In [185]:
p6.quad(

    top=hist,

    bottom=0,

    left=edges[:-1],

    right=edges[1:],

    fill_color="#8B5CF6",

    line_color="white"

)

GlyphRenderer(id='p2713', ...)

In [188]:
p7 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="Volatility Risk Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [189]:
p7.line(

    data['Date'],

    data['Volatility'],

    line_width=3,

    color="#EF4444"

)

GlyphRenderer(id='p2773', ...)

In [190]:
hover = HoverTool(

    tooltips=[

        ("Date", "@Date{%F}"),
        ("Open", "@Open"),
        ("Close", "@Close"),
        ("High", "@High"),
        ("Low", "@Low"),
        ("Volume", "@Volume")

    ],

    formatters={

        '@Date':'datetime'

    }

)

In [191]:
p1.add_tools(hover)

In [192]:
plots = [

    p1,
    p2,
    p3,
    p4,
    p5,
    p6,
    p7

]

In [193]:
for p in plots:

    p.title.text_color = TEXT

    p.xaxis.major_label_text_color = TEXT

    p.yaxis.major_label_text_color = TEXT

    p.xaxis.axis_label_text_color = TEXT

    p.yaxis.axis_label_text_color = TEXT

    p.grid.grid_line_color = GRID

In [194]:
title = Div(text="""

<div style="

background:linear-gradient(
90deg,
#111827,
#2563EB
);

padding:25px;

border-radius:18px;

box-shadow:0px 4px 15px rgba(0,0,0,0.5);

">

<h1 style="

color:white;

text-align:center;

font-size:40px;

">

GOLDMAN SACHS FINANCIAL INTELLIGENCE DASHBOARD

</h1>

<p style="

color:white;

text-align:center;

font-size:18px;

">

Real-Time Investment Banking Analytics |
Risk Intelligence |
Portfolio Monitoring

</p>

</div>

""")

In [195]:
subtitle = Div(text="""

<h2 style="
color:white;
padding:10px;
">

Interactive Investment Banking Analytics Platform

</h2>

""")

In [196]:
divider = Div(text="""

<hr style="
border:2px solid #334155;
">

""")

In [198]:
dashboard = column(

    title,

    subtitle,

    divider,

    row(
        kpi,
        kpi2,
        kpi3
    ),

    row(
        kpi4,
        kpi5,
        kpi6
    ),

    divider,

    row(
        company_selector,
        date_slider
    ),

    row(
        p1,
        p2
    ),

    row(
        p3,
        p4
    ),

    row(
        p5,
        p6
    ),

    p7

)

In [199]:
show(dashboard)

ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p2788', ...)
